# Training Dataset Builder

Engineers features on top of the voyage pipeline output for use in a delay prediction model.

**Features built:**
1. Origin & destination per AIS ping (via `VoyageCreator`)
2. Remaining great-circle distance to destination + fraction of voyage completed
3. Rolling average SOG over the last 6 h and 24 h per vessel
4. Speed deviation: current SOG vs historical lane average SOG
5. **Label** — remaining travel time to arrival (hours)

In [7]:
import sys
sys.path.insert(0, '..')
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

from src.methods import dms_to_dd
from src.voyage_creator import VoyageCreator

## 0. Load data

In [8]:
folder = Path("../data/ais")

df_ais = pd.concat(
    [pd.read_csv(f) for f in sorted(folder.glob("ais-2025-01-0*.csv"))],
    ignore_index=True,
)

df_cargo = df_ais[df_ais["vessel_type"].between(70, 79)].copy()
df_cargo["base_date_time"] = pd.to_datetime(df_cargo["base_date_time"])
df_cargo = df_cargo.sort_values(["mmsi", "base_date_time"]).reset_index(drop=True)

df_ports = pd.read_csv("../data/ports/ports.csv")
df_ports["lat_dd"] = df_ports["latitude"].apply(dms_to_dd)
df_ports["lon_dd"] = df_ports["longitude"].apply(dms_to_dd)
df_ports["geometry"] = df_ports.apply(lambda r: Point(r["lon_dd"], r["lat_dd"]), axis=1)
gdf_ports = gpd.GeoDataFrame(df_ports, geometry="geometry", crs="EPSG:4326")

print(f"Cargo pings : {len(df_cargo):,}")
print(f"Ports       : {len(gdf_ports):,}")

Cargo pings : 4,369,871
Ports       : 657


## Run the voyage pipeline

Detect port visits, label every ping, and build voyage records — same pipeline as `voyage_creator.ipynb`.

In [9]:
creator = VoyageCreator(gdf_ports, radius_nm=10, max_speed_knots=1.5, gap_threshold_h=24)

port_visits         = creator.find_port_visits(df_cargo)
df_labeled          = creator.label_pings(df_cargo, port_visits)
df_labeled, df_voyages = creator.build_voyages(df_labeled, port_visits)

print(f"Port visits   : {len(port_visits):,}")
print(f"Voyages       : {len(df_voyages):,}")
print(f"Labeled pings : {len(df_labeled):,}")

Port visits   : 4,424
Voyages       : 571
Labeled pings : 4,369,871


## Step 1 — Origin & destination per AIS ping

`label_pings()` already added `origin_port`, `destination_port`, and `voyage_id` to every ping.
Here we narrow to **sea pings inside a detected voyage** (both origin and destination are known)
and join `arrival_time` from `df_voyages` so later steps can compute the label.

In [10]:
# Keep only sea pings that belong to a voyage with a known destination
df_sea = df_labeled[
    df_labeled["voyage_id"].notna() & df_labeled["destination_port"].notna()
].copy()

df_sea["voyage_id"] = df_sea["voyage_id"].astype(int)

# Join arrival_time so we can compute remaining travel time later
df_sea = df_sea.merge(
    df_voyages[["voyage_id", "arrival_time"]],
    on="voyage_id",
    how="left",
)

print(f"Sea pings with voyage context : {len(df_sea):,}")
print(f"Unique voyages                : {df_sea['voyage_id'].nunique():,}")
display(
    df_sea[["mmsi", "base_date_time", "sog", "origin_port", "destination_port", "voyage_id", "arrival_time"]]
    .head(10)
)

Sea pings with voyage context : 532,989
Unique voyages                : 523


,mmsi,base_date_time,sog,origin_port,destination_port,voyage_id,arrival_time
0,209277000,2025-01-03 11:45:03,2.3,St. James,Destrehan,0,2025-01-03 14:49:07
1,209277000,2025-01-03 11:46:04,1.9,St. James,Destrehan,0,2025-01-03 14:49:07
2,209277000,2025-01-03 11:47:09,1.7,St. James,Destrehan,0,2025-01-03 14:49:07
3,209277000,2025-01-03 11:48:10,2.1,St. James,Destrehan,0,2025-01-03 14:49:07
4,209277000,2025-01-03 11:49:27,3.7,St. James,Destrehan,0,2025-01-03 14:49:07
5,209277000,2025-01-03 11:50:37,4.7,St. James,Destrehan,0,2025-01-03 14:49:07
6,209277000,2025-01-03 11:51:47,5.8,St. James,Destrehan,0,2025-01-03 14:49:07
7,209277000,2025-01-03 11:52:57,7.5,St. James,Destrehan,0,2025-01-03 14:49:07
8,209277000,2025-01-03 11:54:06,8.8,St. James,Destrehan,0,2025-01-03 14:49:07
9,209277000,2025-01-03 11:55:17,9.8,St. James,Destrehan,0,2025-01-03 14:49:07


## Step 2 — Remaining distance & fraction of voyage completed

For every ping we compute:
- **`remaining_dist_nm`** — great-circle (haversine) distance from the current position to the destination port (nautical miles)
- **`total_dist_nm`** — straight-line distance from origin port to destination port (proxy for voyage length)
- **`fraction_completed`** — `1 - remaining / total`, clipped to [0, 1]

In [11]:
def haversine_nm(lat1, lon1, lat2, lon2):
    """Vectorised great-circle distance in nautical miles."""
    R = 3440.065  # Earth radius in nautical miles
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


# Build port name → (lat_dd, lon_dd) lookup
port_loc = gdf_ports.set_index("portName")[["lat_dd", "lon_dd"]]

# Join destination coordinates
df_sea = df_sea.join(
    port_loc.rename(columns={"lat_dd": "dest_lat", "lon_dd": "dest_lon"}),
    on="destination_port",
)
# Join origin coordinates
df_sea = df_sea.join(
    port_loc.rename(columns={"lat_dd": "origin_lat", "lon_dd": "origin_lon"}),
    on="origin_port",
)

# Distance from current ping to destination
df_sea["remaining_dist_nm"] = haversine_nm(
    df_sea["latitude"],  df_sea["longitude"],
    df_sea["dest_lat"],  df_sea["dest_lon"],
)

# Straight-line voyage length (origin → destination)
df_sea["total_dist_nm"] = haversine_nm(
    df_sea["origin_lat"], df_sea["origin_lon"],
    df_sea["dest_lat"],   df_sea["dest_lon"],
)

# Fraction completed — guard against zero-length voyages (same-port round trips)
df_sea["fraction_completed"] = (
    1 - df_sea["remaining_dist_nm"] / df_sea["total_dist_nm"].replace(0, np.nan)
).clip(0, 1)

display(
    df_sea[["mmsi", "base_date_time", "remaining_dist_nm", "total_dist_nm", "fraction_completed"]]
    .head(10)
    .round(2)
)

/tmp/ipykernel_6067/4254437866.py:45: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,mmsi,base_date_time,remaining_dist_nm,total_dist_nm,fraction_completed
0,209277000,2025-01-03 11:45:03,24.43,24.03,0.0
1,209277000,2025-01-03 11:46:04,24.45,24.03,0.0
2,209277000,2025-01-03 11:47:09,24.45,24.03,0.0
3,209277000,2025-01-03 11:48:10,24.46,24.03,0.0
4,209277000,2025-01-03 11:49:27,24.44,24.03,0.0
5,209277000,2025-01-03 11:50:37,24.42,24.03,0.0
6,209277000,2025-01-03 11:51:47,24.39,24.03,0.0
7,209277000,2025-01-03 11:52:57,24.37,24.03,0.0
8,209277000,2025-01-03 11:54:06,24.34,24.03,0.0
9,209277000,2025-01-03 11:55:17,24.28,24.03,0.0


## Step 3 — Rolling average SOG (6 h and 24 h)

Time-based rolling windows require a `DatetimeIndex`.
We sort by `(mmsi, base_date_time)`, set the index, apply `groupby(mmsi).rolling(window)`,
then reset the index.  The early pings in each window will have fewer observations
(e.g. the first ping's 6 h window contains only itself) — this is expected.

In [12]:
df_sea = df_sea.sort_values(["mmsi", "base_date_time"]).set_index("base_date_time")

df_sea["sog_roll_6h"] = (
    df_sea.groupby("mmsi")["sog"]
    .rolling("6h")
    .mean()
    .reset_index(level=0, drop=True)
)

df_sea["sog_roll_24h"] = (
    df_sea.groupby("mmsi")["sog"]
    .rolling("24h")
    .mean()
    .reset_index(level=0, drop=True)
)

df_sea = df_sea.reset_index()

display(
    df_sea[["mmsi", "base_date_time", "sog", "sog_roll_6h", "sog_roll_24h"]]
    .head(10)
    .round(2)
)

/tmp/ipykernel_6067/525212024.py:22: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,mmsi,base_date_time,sog,sog_roll_6h,sog_roll_24h
0,209277000,2025-01-03 11:45:03,2.3,2.30,2.30
1,209277000,2025-01-03 11:46:04,1.9,2.10,2.10
2,209277000,2025-01-03 11:47:09,1.7,1.97,1.97
3,209277000,2025-01-03 11:48:10,2.1,2.00,2.00
4,209277000,2025-01-03 11:49:27,3.7,2.34,2.34
5,209277000,2025-01-03 11:50:37,4.7,2.73,2.73
6,209277000,2025-01-03 11:51:47,5.8,3.17,3.17
7,209277000,2025-01-03 11:52:57,7.5,3.71,3.71
8,209277000,2025-01-03 11:54:06,8.8,4.28,4.28
9,209277000,2025-01-03 11:55:17,9.8,4.83,4.83


## Step 4 — Speed deviation

`sog_deviation = current_sog - historical_avg_sog_on_lane`

We compute the average SOG per `(mmsi, origin_port, destination_port)` triplet first.
Where a vessel has not appeared on that lane before in the dataset (common given the
data sparsity — 10-day window), we fall back to the fleet-wide average for that lane.

> **Note on data leakage:** in a real training pipeline the historical average would come
> from a separate historical dataset.  Here we use the same dataset as a POC approximation.

In [13]:
LANE_COLS = ["origin_port", "destination_port"]

# Per-vessel per-lane average
vessel_lane_avg = (
    df_sea.groupby(["mmsi"] + LANE_COLS)["sog"]
    .mean()
    .rename("vessel_lane_avg_sog")
    .reset_index()
)

# Fleet-wide per-lane average (fallback)
lane_avg = (
    df_sea.groupby(LANE_COLS)["sog"]
    .mean()
    .rename("lane_avg_sog")
    .reset_index()
)

df_sea = (
    df_sea
    .merge(vessel_lane_avg, on=["mmsi"] + LANE_COLS, how="left")
    .merge(lane_avg,        on=LANE_COLS,             how="left")
)

# Use vessel-level where available, fall back to lane-level
df_sea["hist_avg_sog"]  = df_sea["vessel_lane_avg_sog"].fillna(df_sea["lane_avg_sog"])
df_sea["sog_deviation"] = df_sea["sog"] - df_sea["hist_avg_sog"]
df_sea = df_sea.drop(columns=["vessel_lane_avg_sog", "lane_avg_sog"])

display(
    df_sea[["mmsi", "base_date_time", "sog", "hist_avg_sog", "sog_deviation"]]
    .head(10)
    .round(2)
)

/tmp/ipykernel_6067/708303855.py:33: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,mmsi,base_date_time,sog,hist_avg_sog,sog_deviation
0,209277000,2025-01-03 11:45:03,2.3,12.38,-10.08
1,209277000,2025-01-03 11:46:04,1.9,12.38,-10.48
2,209277000,2025-01-03 11:47:09,1.7,12.38,-10.68
3,209277000,2025-01-03 11:48:10,2.1,12.38,-10.28
4,209277000,2025-01-03 11:49:27,3.7,12.38,-8.68
5,209277000,2025-01-03 11:50:37,4.7,12.38,-7.68
6,209277000,2025-01-03 11:51:47,5.8,12.38,-6.58
7,209277000,2025-01-03 11:52:57,7.5,12.38,-4.88
8,209277000,2025-01-03 11:54:06,8.8,12.38,-3.58
9,209277000,2025-01-03 11:55:17,9.8,12.38,-2.58


## Step 5 — Label: remaining travel time

`remaining_travel_hours = arrival_time - base_date_time`

Pings logged after the recorded `arrival_time` (a data artefact from the gap-based
window detection) yield a negative label and are dropped.

In [14]:
df_sea["remaining_travel_hours"] = (
    (df_sea["arrival_time"] - df_sea["base_date_time"]).dt.total_seconds() / 3600
)

n_neg = (df_sea["remaining_travel_hours"] < 0).sum()
if n_neg:
    print(f"Dropping {n_neg:,} pings with negative remaining travel time (post-arrival artefacts)")
    df_sea = df_sea[df_sea["remaining_travel_hours"] >= 0]

display(
    df_sea[["mmsi", "base_date_time", "arrival_time", "remaining_travel_hours"]]
    .head(10)
    .round(2)
)

/tmp/ipykernel_6067/213010261.py:13: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,mmsi,base_date_time,arrival_time,remaining_travel_hours
0,209277000,2025-01-03 11:45:03,2025-01-03 14:49:07,3.07
1,209277000,2025-01-03 11:46:04,2025-01-03 14:49:07,3.05
2,209277000,2025-01-03 11:47:09,2025-01-03 14:49:07,3.03
3,209277000,2025-01-03 11:48:10,2025-01-03 14:49:07,3.02
4,209277000,2025-01-03 11:49:27,2025-01-03 14:49:07,2.99
5,209277000,2025-01-03 11:50:37,2025-01-03 14:49:07,2.98
6,209277000,2025-01-03 11:51:47,2025-01-03 14:49:07,2.96
7,209277000,2025-01-03 11:52:57,2025-01-03 14:49:07,2.94
8,209277000,2025-01-03 11:54:06,2025-01-03 14:49:07,2.92
9,209277000,2025-01-03 11:55:17,2025-01-03 14:49:07,2.90


## Training dataset summary

In [15]:
FEATURE_COLS = [
    "mmsi", "base_date_time", "latitude", "longitude", "sog",
    "origin_port", "destination_port", "voyage_id",
    "remaining_dist_nm", "total_dist_nm", "fraction_completed",
    "sog_roll_6h", "sog_roll_24h",
    "hist_avg_sog", "sog_deviation",
    "remaining_travel_hours",  # label
]

df_train = df_sea[FEATURE_COLS].reset_index(drop=True)

print(f"Training dataset shape : {df_train.shape}")
print(f"Unique voyages         : {df_train['voyage_id'].nunique():,}")
print(f"Unique vessels         : {df_train['mmsi'].nunique():,}")
print()
print("Null counts:")
display(df_train.isnull().sum().to_frame("nulls").query("nulls > 0"))
print()
print("Label statistics (remaining_travel_hours):")
display(df_train["remaining_travel_hours"].describe().round(2))
print()
display(df_train.head(10).round(2))

Training dataset shape : (597872, 16)
Unique voyages         : 523
Unique vessels         : 378

Null counts:


,nulls
sog,3929
fraction_completed,51218
sog_roll_6h,3915
sog_roll_24h,3915
hist_avg_sog,3915
sog_deviation,3929



Label statistics (remaining_travel_hours):


count    597872.00
mean         30.23
std          28.22
min           0.01
25%           8.00
50%          22.33
75%          43.37
max         157.96
Name: remaining_travel_hours, dtype: float64

/tmp/ipykernel_6067/2956047222.py:22: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_train.head(10).round(2))


,mmsi,base_date_time,latitude,longitude,sog,origin_port,destination_port,voyage_id,remaining_dist_nm,total_dist_nm,fraction_completed,sog_roll_6h,sog_roll_24h,hist_avg_sog,sog_deviation,remaining_travel_hours
0,209277000,2025-01-03 11:45:03,30.01,-90.83,2.3,St. James,Destrehan,0,24.43,24.03,0.0,2.30,2.30,12.38,-10.08,3.07
1,209277000,2025-01-03 11:46:04,30.01,-90.83,1.9,St. James,Destrehan,0,24.45,24.03,0.0,2.10,2.10,12.38,-10.48,3.05
2,209277000,2025-01-03 11:47:09,30.01,-90.83,1.7,St. James,Destrehan,0,24.45,24.03,0.0,1.97,1.97,12.38,-10.68,3.03
3,209277000,2025-01-03 11:48:10,30.00,-90.83,2.1,St. James,Destrehan,0,24.46,24.03,0.0,2.00,2.00,12.38,-10.28,3.02
4,209277000,2025-01-03 11:49:27,30.00,-90.83,3.7,St. James,Destrehan,0,24.44,24.03,0.0,2.34,2.34,12.38,-8.68,2.99
5,209277000,2025-01-03 11:50:37,30.00,-90.83,4.7,St. James,Destrehan,0,24.42,24.03,0.0,2.73,2.73,12.38,-7.68,2.98
6,209277000,2025-01-03 11:51:47,30.00,-90.83,5.8,St. James,Destrehan,0,24.39,24.03,0.0,3.17,3.17,12.38,-6.58,2.96
7,209277000,2025-01-03 11:52:57,30.00,-90.83,7.5,St. James,Destrehan,0,24.37,24.03,0.0,3.71,3.71,12.38,-4.88,2.94
8,209277000,2025-01-03 11:54:06,30.00,-90.83,8.8,St. James,Destrehan,0,24.34,24.03,0.0,4.28,4.28,12.38,-3.58,2.92
9,209277000,2025-01-03 11:55:17,29.99,-90.83,9.8,St. James,Destrehan,0,24.28,24.03,0.0,4.83,4.83,12.38,-2.58,2.90


## Model Training (scikit-learn)

Build a `Pipeline` with a `ColumnTransformer` preprocessor and a `GradientBoostingRegressor`
to predict **remaining travel time** from the engineered features.

Split is done **by voyage** (not by ping) to prevent pings from the same voyage appearing
in both train and test sets.

In [16]:
NUM_FEATURES = [
    "sog", "sog_roll_6h", "sog_roll_24h", "sog_deviation",
    "remaining_dist_nm", "fraction_completed",
]
CAT_FEATURES = ["origin_port", "destination_port"]
TARGET       = "remaining_travel_hours"

# Drop rows where any feature or label is NaN
df_model = df_train[NUM_FEATURES + CAT_FEATURES + ["voyage_id", TARGET]].dropna()

print(f"Rows after dropna : {len(df_model):,}  (from {len(df_train):,})")

Rows after dropna : 546,640  (from 597,872)


In [17]:
# Split by voyage so no voyage straddles train and test
voyage_ids = df_model["voyage_id"].unique()
train_ids, test_ids = train_test_split(voyage_ids, test_size=0.2, random_state=42)

train_df = df_model[df_model["voyage_id"].isin(train_ids)]
test_df  = df_model[df_model["voyage_id"].isin(test_ids)]

X_train, y_train = train_df[NUM_FEATURES + CAT_FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[NUM_FEATURES + CAT_FEATURES],  test_df[TARGET]

print(f"Train : {len(X_train):,} pings from {len(train_ids)} voyages")
print(f"Test  : {len(X_test):,} pings from {len(test_ids)} voyages")

Train : 443,816 pings from 377 voyages
Test  : 102,824 pings from 95 voyages


In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
    )),
])

model.fit(X_train, y_train)
print("Model trained.")

In [ ]:
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f} h")
print(f"RMSE : {rmse:.2f} h")
print(f"R²   : {r2:.3f}")

In [ ]:
cat_feature_names = (
    model.named_steps["preprocessor"]
    .named_transformers_["cat"]
    .get_feature_names_out(CAT_FEATURES)
    .tolist()
)

importances = pd.Series(
    model.named_steps["regressor"].feature_importances_,
    index=NUM_FEATURES + cat_feature_names,
).sort_values(ascending=False)

print("Top 20 feature importances:")
display(importances.head(20).round(4).to_frame("importance"))

In [ ]:
model_path = Path("../models/delay_predictor.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(model, model_path)
print(f"Model saved to {model_path.resolve()}")